# Qwen3 MLX 部署与交互教程

本教程演示如何在 Apple Silicon Mac 上使用 **MLX** 框架部署 Qwen3 大模型，包括：

1. 环境准备与依赖安装
2. 模型下载
3. 模型加载与文本生成
4. 流式输出

> **硬件要求**：Apple Silicon Mac（M1/M2/M3/M4 系列）  
> **推荐模型**：`Qwen3-8B-4bit`（约需 6GB 内存）
>
> 如需 **交互式聊天界面**，请运行独立脚本：`python Qwen3_gradio.py`

## 1. 环境准备

安装所需依赖（如已安装可跳过）：

In [11]:
!pip install mlx-lm==0.30.2 gradio

## 2. 模型下载

从 `mlx-community` 下载已量化的 MLX 格式模型。首次下载需要一些时间，后续会使用本地缓存。

In [12]:
from huggingface_hub import snapshot_download
import time

# ============================================================
# 📦 配置
# ============================================================
MODEL_NAME = "Qwen3-8B-4bit"       # 模型名称（mlx-community 下的仓库名）
LOCAL_DIR = "../models"             # 本地存放路径
REPO_ID = f"mlx-community/{MODEL_NAME}"  # 仓库ID
# ============================================================

MODEL_PATH = f"{LOCAL_DIR}/{MODEL_NAME}"

print(f"📋 Repo ID:   {REPO_ID}")
print(f"📋 本地路径:  {MODEL_PATH}")
print()

s = time.time()
snapshot_download(repo_id=REPO_ID, local_dir=MODEL_PATH)
print(f"\n✅ 下载完成，耗时 {time.time()-s:.2f} 秒")

📋 Repo ID:   mlx-community/Qwen3-8B-4bit
📋 本地路径:  ../models/Qwen3-8B-4bit



Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]


✅ 下载完成，耗时 1.54 秒


## 3. 模型加载与文本生成

使用 `mlx-lm` 加载模型并进行一次性文本生成，适合快速测试。

In [13]:
import mlx.core as mx
from mlx_lm import load, generate
from mlx_lm.sample_utils import make_sampler
import time

# 加载模型
print(f"🚀 正在加载模型 {MODEL_NAME} ...")
s = time.time()
model, tokenizer = load(MODEL_PATH)
mx.eval()
print(f"✅ 模型加载完成，耗时 {time.time()-s:.2f} 秒")

🚀 正在加载模型 Qwen3-8B-4bit ...
✅ 模型加载完成，耗时 2.03 秒


In [14]:
# 构建对话 prompt
question = "请用一句话解释什么是人工智能？"

messages = [
    {"role": "system", "content": "你是一个智能助手。"},
    {"role": "user", "content": question},
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

# 一次性生成
sampler = make_sampler(temp=0.7, top_p=0.8)

s_time = time.time()
response = generate(
    model, tokenizer,
    prompt=prompt,
    max_tokens=256,
    sampler=sampler,
    verbose=False,
)
mx.eval()
gen_time = time.time() - s_time

# 统计
prompt_tokens = len(tokenizer.encode(prompt))
response_tokens = len(tokenizer.encode(response))

print("🤖 模型回复\n" + "=" * 50)
print(response)
print("=" * 50)
print(f"\n📊 性能统计")
print(f"  Prompt tokens:  {prompt_tokens}")
print(f"  生成 tokens:    {response_tokens}")
print(f"  生成耗时:       {gen_time:.2f}s")
print(f"  生成速度:       {response_tokens / gen_time:.1f} tokens/s")
print(f"  峰值内存:       {mx.get_peak_memory() / 1024**3:.2f} GB")

🤖 模型回复
<think>
好的，用户让我用一句话解释什么是人工智能。首先，我需要确保定义准确且简洁。人工智能通常被描述为模拟人类智能的系统，但可能需要更具体一些。

用户可能是学生或对技术不太熟悉的人，所以需要避免专业术语。要涵盖关键点：模拟人类智能、学习和适应能力、应用领域。

可能需要提到机器学习、数据分析，但保持简短。比如“人工智能是模拟人类智能的计算机系统，能够学习、推理和执行任务。”这样既涵盖核心要素，又容易理解。

还要检查是否有遗漏的重要概念，比如是否要区分弱人工智能和强人工智能，但用户只要一句话，所以保持简洁。确认没有误解，确保句子流畅自然。
</think>

人工智能是模拟人类智能的计算机系统，能够通过学习、推理和适应来执行复杂任务。

📊 性能统计
  Prompt tokens:  25
  生成 tokens:    173
  生成耗时:       15.59s
  生成速度:       11.1 tokens/s
  峰值内存:       8.58 GB
